<a href="https://colab.research.google.com/github/Krishnapal1900/soft-computing-project/blob/main/Traffic_Light_Controller_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q gradio scikit-fuzzy numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# FUZZY VARIABLES
traffic_density = ctrl.Antecedent(np.arange(0,101,1), 'traffic_density')                              #universe of discourse (range of values)
waiting_time    = ctrl.Antecedent(np.arange(0,61,1), 'waiting_time')
weather         = ctrl.Antecedent(np.arange(0,11,1), 'weather')

green_time      = ctrl.Consequent(np.arange(10,121,1), 'green_time')


# MEMBERSHIP FUNCTIONS
# Traffic Density
traffic_density['low']    = fuzz.trimf(traffic_density.universe,[0,0,40])
traffic_density['medium'] = fuzz.trimf(traffic_density.universe,[20,50,80])
traffic_density['high']   = fuzz.trimf(traffic_density.universe,[60,100,100])

# Waiting Time
waiting_time['short']  = fuzz.trimf(waiting_time.universe,[0,0,20])
waiting_time['medium'] = fuzz.trimf(waiting_time.universe,[10,30,50])
waiting_time['long']   = fuzz.trimf(waiting_time.universe,[40,60,60])

# Weather
weather['clear']  = fuzz.trimf(weather.universe,[0,0,3])
weather['normal'] = fuzz.trimf(weather.universe,[2,5,8])
weather['bad']    = fuzz.trimf(weather.universe,[6,10,10])

# Output Green Time
green_time['short']  = fuzz.trimf(green_time.universe,[10,20,40])
green_time['medium'] = fuzz.trimf(green_time.universe,[35,55,75])
green_time['long']   = fuzz.trimf(green_time.universe,[70,95,120])


# RULES

rules = [

ctrl.Rule(traffic_density['low'] & waiting_time['short'], green_time['short']),
ctrl.Rule(traffic_density['low'] & waiting_time['medium'], green_time['medium']),
ctrl.Rule(traffic_density['low'] & waiting_time['long'], green_time['medium']),

ctrl.Rule(traffic_density['medium'] & waiting_time['short'], green_time['medium']),
ctrl.Rule(traffic_density['medium'] & waiting_time['medium'], green_time['medium']),
ctrl.Rule(traffic_density['medium'] & waiting_time['long'], green_time['long']),

ctrl.Rule(traffic_density['high'] & waiting_time['short'], green_time['medium']),
ctrl.Rule(traffic_density['high'] & waiting_time['medium'], green_time['long']),
ctrl.Rule(traffic_density['high'] & waiting_time['long'], green_time['long']),

ctrl.Rule(weather['bad'], green_time['long']),
ctrl.Rule(weather['clear'] & traffic_density['low'], green_time['short'])
]

system = ctrl.ControlSystem(rules)

# MAIN FUNCTION

def smart_traffic(
    density,
    wait,
    weather_input,
    emergency,
    pedestrian,
    time_day,
    cars,
    buses,
    trucks,
    bikes,
    north,
    south,
    east,
    west,
    vip,
    accident
):

    yellow_sec = 4                                    # Fixed yellow light time
    total_cycle_max = 180                             # Max cycle time for the intersection (can be adjusted)

    # Emergency Override
    if emergency == "Yes":
        msg = "🚑 Emergency Vehicle Detected!\nImmediate GREEN Signal Activated."
        sec = 120
        red_sec = total_cycle_max - sec - yellow_sec
        if red_sec < 0: red_sec = 0

    # VIP Override
    elif vip == "Yes":
        msg = "🚓 VIP / Police Override Enabled.\nPriority GREEN Signal Activated."
        sec = 100
        red_sec = total_cycle_max - sec - yellow_sec
        if red_sec < 0: red_sec = 0

    else:
        sim = ctrl.ControlSystemSimulation(system)

        # Lane Density Avg
        lane_avg = (north + south + east + west) / 4

        # Vehicle Weight
        vehicle_load = cars + (buses*3) + (trucks*4) + bikes*0.5

        final_density = min(100, (density*0.5) + (lane_avg*0.3) + (vehicle_load*0.2))

        # Time of Day Adjustment
        if time_day == "Morning Rush":
            final_density += 10
        elif time_day == "Night":
            final_density -= 10

        final_density = max(0, min(100, final_density))

        sim.input['traffic_density'] = final_density
        sim.input['waiting_time'] = wait
        sim.input['weather'] = weather_input
        sim.compute()

        sec = sim.output['green_time']

        # Pedestrian
        if pedestrian == "Yes":
            sec = max(20, sec - 10)

        # Accident
        if accident == "Yes":
            sec += 20

        # Calculate red time for THIS specific light
        red_sec = total_cycle_max - sec - yellow_sec
        if red_sec < 0: # Ensure red_sec is not negative, happens if green+yellow > total_cycle_max
            red_sec = 0 # Set to minimum if calculation goes below zero

    msg = f"""
🚦 Smart Traffic Decision

Green Signal Time: {sec:.2f} sec
Yellow Signal Time: {yellow_sec:.2f} sec
Red Signal Time: {red_sec:.2f} sec

Adjusted Density: {final_density:.2f}%

Pedestrian Request: {pedestrian}
Accident Mode: {accident}
Time Slot: {time_day}
"""

    # Chart
    fig, ax = plt.subplots(figsize=(8,3))
    ax.bar(["Green Time", "Yellow Time", "Red Time"], [sec, yellow_sec, red_sec], color=['green', 'yellow', 'red'])
    ax.set_ylim(0,total_cycle_max) # Set ylim to total cycle max
    ax.set_ylabel("Seconds")
    ax.set_title("Traffic Signal Timing")

    return msg, fig, sec, yellow_sec, red_sec # Return these values

# ==========================================================
# GRADIO UI
# ==========================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo: # Moved theme parameter here

    gr.Markdown("# 🚦 Smart Traffic Light Controller")
    gr.Markdown("### Advanced Fuzzy Logic Based Mini Project")

    with gr.Tab("Main Control"):
        density = gr.Slider(0,100,50,label="Traffic Density %")
        wait = gr.Slider(0,60,20,label="Waiting Time (sec)")
        weather_input = gr.Slider(0,10,2,label="Weather Condition (0 Clear - 10 Bad)")
        time_day = gr.Dropdown(
            ["Morning Rush","Normal Day","Night"],
            value="Normal Day",
            label="Time of Day"
        )

    with gr.Tab("Priority Inputs"):
        emergency = gr.Radio(["No","Yes"], value="No", label="🚑 Emergency Vehicle")
        pedestrian = gr.Radio(["No","Yes"], value="No", label="🚶 Pedestrian Crossing")
        vip = gr.Radio(["No","Yes"], value="No", label="🚓 VIP Route / Police Override")
        accident = gr.Radio(["No","Yes"], value="No", label="🚦 Accident Detected")

    with gr.Tab("Vehicle Count"):
        cars = gr.Slider(0,100,20,label="Cars")
        buses = gr.Slider(0,30,2,label="Buses")
        trucks = gr.Slider(0,20,1,label="Trucks")
        bikes = gr.Slider(0,100,10,label="Bikes")

    with gr.Tab("Lane Camera Count"):
        north = gr.Slider(0,100,20,label="North Lane")
        south = gr.Slider(0,100,20,label="South Lane")
        east = gr.Slider(0,100,20,label="East Lane")
        west = gr.Slider(0,100,20,label="West Lane")

    btn = gr.Button("🚦 Run Smart Controller")

    output_msg = gr.Textbox(label="Traffic Result")
    output_graph = gr.Plot(label="Timing Graph")
    output_green = gr.Number(label="Green Time (sec)")
    output_yellow = gr.Number(label="Yellow Time (sec)")
    output_red = gr.Number(label="Red Time (sec)")

    btn.click(
        smart_traffic,
        inputs=[
            density, wait, weather_input,
            emergency, pedestrian, time_day,
            cars, buses, trucks, bikes,
            north, south, east, west,
            vip, accident
        ],
        outputs=[output_msg, output_graph, output_green, output_yellow, output_red]
    )

demo.launch(share=True)

/tmp/ipykernel_6376/707147356.py:164: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo: # Moved theme parameter here


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://62c280154dc5fc0218.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
